In [22]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [5]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS"
vix = pd.read_csv(url, na_values = ".")
vix.columns = ['date', "VIXCLS"]
vix['date'] = pd.to_datetime(vix['date'])
vix = vix.set_index('date')
vix.to_csv("VIXCLS.csv")
print(vix.head(3))
print(vix.tail(3))
print(len(vix), "rows,", vix["VIXCLS"].isna().sum(), "missing")
print(vix["VIXCLS"].describe())

            VIXCLS
date              
1990-01-02   17.24
1990-01-03   18.19
1990-01-04   19.22
            VIXCLS
date              
2026-09-29   16.04
2026-09-30   16.34
2026-10-01   16.39
9588 rows, 302 missing
count    9286.000000
mean       19.423616
std         7.715535
min         9.140000
25%        13.992500
50%        17.570000
75%        22.677500
max        82.690000
Name: VIXCLS, dtype: float64


In [8]:
daily = pd.read_csv("../../data/ff_daily_garchdata_long.csv")
print(daily.head(8))
print(daily.dtypes)
print(daily.iloc[:, 0].nunique(), "unique values in first column")

         Date  series     ret    month
0  01-07-1963  Mkt-RF -0.0067  1963-07
1  02-07-1963  Mkt-RF  0.0079  1963-07
2  03-07-1963  Mkt-RF  0.0063  1963-07
3  05-07-1963  Mkt-RF  0.0040  1963-07
4  08-07-1963  Mkt-RF -0.0063  1963-07
5  09-07-1963  Mkt-RF  0.0045  1963-07
6  10-07-1963  Mkt-RF -0.0018  1963-07
7  11-07-1963  Mkt-RF -0.0016  1963-07
Date       object
series     object
ret       float64
month      object
dtype: object
15897 unique values in first column


In [10]:
mkt = daily[daily["series"] == "Mkt-RF"].copy()
mkt["Date"] = pd.to_datetime(mkt["Date"], format="%d-%m-%Y")
r = mkt.set_index("Date")["ret"].sort_index().loc["1990-01-01":]

v = vix["VIXCLS"].reindex(r.index).ffill(limit=3)   # VIX on each return day, small gaps filled

print(len(r), r.index.min().date(), r.index.max().date())
print("return days with no VIX:", v.isna().sum())
print(v[v.isna()])

print("corr(|r_t|, VIX_t)  :", r.abs().corr(v))
print("corr(|r_t|, VIX_t-1):", r.abs().corr(v.shift(1)))

9233 1990-01-02 2026-08-31
return days with no VIX: 0
Series([], Name: VIXCLS, dtype: float64)
corr(|r_t|, VIX_t)  : 0.5553244093396328
corr(|r_t|, VIX_t-1): 0.5264414147873024


In [15]:
r_pct = r.to_numpy() * 100
x = (v**2/252).to_numpy()
print(x, r_pct)

[1.17943492 1.3130004  1.46590635 ... 0.83547659 0.82628929 0.88335873] [ 1.44 -0.06 -0.71 ...  0.72 -0.34 -0.33]


In [16]:
def variance_path(theta,r,x):
    omega, alpha, beta, delta = theta
    n = len(r)
    h = np.empty(n)
    h[0] = np.var(r)
    for t in range(1,n):
        h[t] = omega + alpha * r[t-1] ** 2 + beta * h[t-1] + delta * x[t-1]
    return h


In [18]:
def negative_loglikelihood(theta, r, x):
    omega, alpha, beta, delta = theta
    if alpha + beta >= 0.999 or omega <= 0 or alpha < 0 or beta < 0 or delta < 0:
        return 1e10
    h = variance_path(theta,r,x)
    if (h<= 0).any():
        return 1e10

    log_like = (1/2)*np.sum(np.log(h[1:]) + r[1:] ** 2/ h [1:])

    return log_like


In [20]:
def simulate(theta, x, seed = 0):
    omega, alpha, beta, delta = theta
    rng = np.random.default_rng(seed)
    n = len(x)
    z = rng.standard_normal(n)
    h = np.empty(n)
    r = np.empty(n)
    h[0] = (omega + delta * x.mean())/(1 - alpha - beta)
    r[0] = np.sqrt(h[0]) * z[0]

    for t in range(1,n):
        h[t] = omega + alpha * r[t-1] ** 2 + beta * h[t-1] + delta * x[t-1]
        r[t] = np.sqrt(h[t]) * z[t]

    return r,h


In [24]:
def fit(r,x):
    start = [0.05, 0.05, 0.90, 0.01]
    bounds = [(1e-6,None), (0,1), (0,1), (0, None)]
    res = minimize(fun = lambda p: negative_loglikelihood(p,r,x)/len(r), x0 = start,  method = "L-BFGS-B", bounds = bounds)
    return res

In [26]:
res = fit(r_pct, x)
print(res.x.round(4), res.success, "persistence", round(res.x[1] + res.x[2], 3))
print("per-obs NLL:", res.fun)

[0.     0.0335 0.1651 0.4925] True persistence 0.199
per-obs NLL: 0.3958398076459012


In [29]:
def forecast(theta, r, x, horizon = 21):
    omega, alpha, beta, delta = theta
    h = variance_path(theta, r, x)
    path = np.empty(horizon)
    path[0] = omega + alpha * r[-1] ** 2 + beta * h[-1] + delta * x[-1]
    for k in range(1,horizon):
        path[k] = (omega + delta *  x[-1]) + (alpha + beta) * path[k-1]
    return path

In [31]:
r_t = np.array([1, -2, 0.5]); x_t = np.array([2, 2, 2])
print(forecast((0.1, 0.1, 0.8, 0.5), r_t, x_t)[:3])      # expect [3.9748 4.6773 5.3096]

[3.97477778 4.6773     5.30957   ]


In [32]:
# 1. closed form
p = forecast(res.x, r_pct, x)
rho = res.x[1] + res.x[2]
h_ss = (res.x[0] + res.x[3] * x[-1]) / (1 - rho)
print(np.allclose(p, h_ss + rho ** np.arange(21) * (p[0] - h_ss)))   # expect True

# 2. units
print(p.sum() / 1e4)    # roughly 0.001 to 0.003

True
0.0011370889494052108
